# 🤖 Modelado y Evaluación
## Online Shoppers Purchasing Intention Dataset

**Objetivo**: Entrenar y evaluar modelos de clasificación para predecir intención de compra.

**Modelos a evaluar**:
1. Logistic Regression (baseline)
2. Decision Tree
3. Random Forest
4. Gradient Boosting
5. XGBoost
6. LightGBM

**Métricas**: Accuracy, Precision, Recall, F1-Score, ROC-AUC  
**Optimización**: GridSearchCV con validación cruzada

---

## 1️⃣ Importar Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from datetime import datetime
warnings.filterwarnings('ignore')

# Configurar estilo de gráficas
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Modelos de clasificación
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Evaluación
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve, auc,
                             precision_recall_curve, matthews_corrcoef)
from sklearn.model_selection import cross_val_score, GridSearchCV, learning_curve

# Serialización
import joblib

# Crear directorio de reportes si no existe
os.makedirs('../E_reports', exist_ok=True)
os.makedirs('../reports/figures', exist_ok=True)

print("✅ Librerías importadas correctamente")
print(f"📁 Directorio de reportes creado: ../reports/")
print(f"📁 Directorio de figuras creado: ../reports/figures/")

## 2️⃣ Cargar Datos Procesados

In [ ]:
# Cargar conjuntos de entrenamiento y prueba
X_train = np.load('../data/02_processed/X_train_balanced.npy')
y_train = np.load('../data/02_processed/y_train_balanced.npy')
X_test = np.load('../data/02_processed/X_test.npy')
y_test = np.load('../data/02_processed/y_test.npy')

# Cargar nombres de features
with open('../data/02_processed/feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]

print(f"✅ Datos cargados:")
print(f"  - X_train: {X_train.shape}")
print(f"  - X_test: {X_test.shape}")
print(f"  - Features: {len(feature_names)}")
print(f"  - Balance train: {(y_train==1).sum()/len(y_train)*100:.1f}% Compra")
print(f"  - Balance test: {(y_test==1).sum()/len(y_test)*100:.1f}% Compra")

## 3️⃣ Función de Evaluación de Modelos

Función estándar para evaluar todos los modelos con las mismas métricas.

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """
    Evalúa modelo de clasificación con múltiples métricas.
    
    Args:
        model: Modelo de scikit-learn/xgboost/lightgbm
        X_train, y_train: Datos de entrenamiento
        X_test, y_test: Datos de prueba
        model_name: Nombre del modelo para identificación
    
    Returns:
        results: Diccionario con métricas
        model: Modelo entrenado
        y_pred_test: Predicciones en test
        y_proba_test: Probabilidades en test
    """
    # Entrenar modelo con datos balanceados
    model.fit(X_train, y_train)
    
    # Predicciones
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Probabilidades para ROC-AUC
    if hasattr(model, 'predict_proba'):
        y_proba_test = model.predict_proba(X_test)[:, 1]
    else:
        y_proba_test = y_pred_test
    
    # Calcular métricas
    results = {
        'Model': model_name,
        'Train_Accuracy': accuracy_score(y_train, y_pred_train),
        'Test_Accuracy': accuracy_score(y_test, y_pred_test),
        'Precision': precision_score(y_test, y_pred_test),
        'Recall': recall_score(y_test, y_pred_test),
        'F1_Score': f1_score(y_test, y_pred_test),
        'ROC_AUC': roc_auc_score(y_test, y_proba_test)
    }
    
    return results, model, y_pred_test, y_proba_test

print("✅ Función de evaluación definida")

## 4️⃣ Entrenamiento de Modelos Base

Entrenamos 6 modelos con hiperparámetros por defecto para establecer baseline.

In [ ]:
# Diccionario de modelos a entrenar (ampliado)
models = {
    # Modelos Lineales
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    
    # Árboles
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    
    # Ensemble - Bagging
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, max_depth=15, random_state=42),
    
    # Ensemble - Boosting
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=6, random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(n_estimators=100, max_depth=6, random_state=42, verbose=0),
    
    # Otros
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42)
}

# Entrenar y evaluar cada modelo
results_list = []
trained_models = {}
all_predictions = {}
all_probabilities = {}

print("🚀 Entrenando 12 modelos de clasificación...")
print("="*100)

for name, model in models.items():
    print(f"\n🔹 {name}")
    start_time = datetime.now()
    
    # Evaluar modelo
    results, trained_model, y_pred, y_proba = evaluate_model(
        model, X_train, y_train, X_test, y_test, name
    )
    
    elapsed_time = (datetime.now() - start_time).total_seconds()
    results['Training_Time'] = elapsed_time
    
    # Guardar resultados
    results_list.append(results)
    trained_models[name] = trained_model
    all_predictions[name] = y_pred
    all_probabilities[name] = y_proba
    
    # Mostrar métricas clave
    print(f"  Train Acc: {results['Train_Accuracy']:.4f}")
    print(f"  Test Acc:  {results['Test_Accuracy']:.4f}")
    print(f"  Precision: {results['Precision']:.4f}")
    print(f"  Recall:    {results['Recall']:.4f}")
    print(f"  F1-Score:  {results['F1_Score']:.4f}")
    print(f"  ROC-AUC:   {results['ROC_AUC']:.4f}")
    print(f"  Time:      {elapsed_time:.2f}s")

print("\n✅ Entrenamiento de todos los modelos completado")

## 5️⃣ Comparación de Modelos

In [ ]:
# Crear DataFrame con resultados
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values('F1_Score', ascending=False)

print("📊 Comparación de Modelos:")
print("="*100)
print(results_df.to_string(index=False))

# Identificar mejor modelo
best_model_name = results_df.iloc[0]['Model']
print(f"\n🏆 Mejor modelo: {best_model_name}")
print(f"   F1-Score: {results_df.iloc[0]['F1_Score']:.4f}")

In [ ]:
# Visualización de comparación completa
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

metrics = ['Test_Accuracy', 'Precision', 'Recall', 'F1_Score', 'ROC_AUC', 'Training_Time']
titles = ['Test Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Training Time (s)']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for idx, (metric, title, color) in enumerate(zip(metrics, titles, colors)):
    row = idx // 3
    col = idx % 3
    
    ax = axes[row, col]
    data = results_df.sort_values(metric, ascending=(metric != 'Training_Time'))
    
    bars = ax.barh(data['Model'], data[metric], color=color, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Score' if metric != 'Training_Time' else 'Seconds', fontsize=11)
    ax.set_title(f'{title} by Model', fontsize=13, fontweight='bold')
    
    if metric != 'Training_Time':
        ax.set_xlim([0, 1])
    
    ax.grid(axis='x', alpha=0.3)
    
    # Añadir valores
    for i, v in enumerate(data[metric]):
        if metric == 'Training_Time':
            ax.text(v + 0.5, i, f'{v:.1f}s', va='center', fontsize=9)
        else:
            ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.suptitle('📊 Análisis Comparativo Completo de 12 Modelos', y=1.002, fontsize=16, fontweight='bold')

# Guardar figura
plt.savefig('../reports/figures/01_model_comparison.png', dpi=300, bbox_inches='tight')
print("✅ Figura guardada: reports/figures/01_model_comparison.png")
plt.show()

In [ ]:
# Análisis de Overfitting/Underfitting
fig, ax = plt.subplots(figsize=(14, 8))

train_accs = results_df['Train_Accuracy'].values
test_accs = results_df['Test_Accuracy'].values
models_names = results_df['Model'].values

x = np.arange(len(models_names))
width = 0.35

bars1 = ax.bar(x - width/2, train_accs, width, label='Train Accuracy', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, test_accs, width, label='Test Accuracy', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('🔍 Train vs Test Accuracy - Análisis de Overfitting', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models_names, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Añadir diferencias
for i, (train, test) in enumerate(zip(train_accs, test_accs)):
    diff = train - test
    color = 'red' if diff > 0.05 else 'green'
    ax.text(i, max(train, test) + 0.01, f'Δ{diff:.3f}', 
            ha='center', fontsize=8, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/02_overfitting_analysis.png', dpi=300, bbox_inches='tight')
print("✅ Figura guardada: reports/figures/02_overfitting_analysis.png")
plt.show()

In [ ]:
# Matrices de Confusión para todos los modelos
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.ravel()

for idx, (name, y_pred) in enumerate(all_predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    
    # Calcular métricas adicionales
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    
    # Crear heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[idx],
                xticklabels=['No Compra', 'Compra'],
                yticklabels=['No Compra', 'Compra'])
    
    axes[idx].set_title(f'{name}\nSensitivity: {sensitivity:.3f} | Specificity: {specificity:.3f}', 
                        fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('True Label', fontsize=10)
    axes[idx].set_xlabel('Predicted Label', fontsize=10)

plt.suptitle('📊 Matrices de Confusión - Todos los Modelos', fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
plt.savefig('../reports/figures/03_confusion_matrices.png', dpi=300, bbox_inches='tight')
print("✅ Figura guardada: reports/figures/03_confusion_matrices.png")
plt.show()

In [ ]:
# Curvas ROC para todos los modelos
fig, ax = plt.subplots(figsize=(12, 10))

colors = plt.cm.tab20(np.linspace(0, 1, len(all_probabilities)))

for (name, y_proba), color in zip(all_probabilities.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    
    ax.plot(fpr, tpr, lw=2, color=color, 
            label=f'{name} (AUC = {roc_auc:.3f})')

# Línea diagonal de referencia
ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier (AUC = 0.500)')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('📈 Curvas ROC - Comparación de Todos los Modelos', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/04_roc_curves.png', dpi=300, bbox_inches='tight')
print("✅ Figura guardada: reports/figures/04_roc_curves.png")
plt.show()

In [ ]:
# Curvas Precision-Recall para todos los modelos
fig, ax = plt.subplots(figsize=(12, 10))

colors = plt.cm.tab20(np.linspace(0, 1, len(all_probabilities)))

for (name, y_proba), color in zip(all_probabilities.items(), colors):
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(recall, precision)
    
    ax.plot(recall, precision, lw=2, color=color, 
            label=f'{name} (AUC = {pr_auc:.3f})')

# Línea base (proporción de clase positiva)
baseline = (y_test == 1).sum() / len(y_test)
ax.plot([0, 1], [baseline, baseline], 'k--', lw=2, 
        label=f'Baseline (Prevalence = {baseline:.3f})')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax.set_title('📈 Curvas Precision-Recall - Comparación de Todos los Modelos', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/05_precision_recall_curves.png', dpi=300, bbox_inches='tight')
print("✅ Figura guardada: reports/figures/05_precision_recall_curves.png")
plt.show()

In [ ]:
# Guardar tabla de resultados en CSV
results_df_sorted = results_df.sort_values('F1_Score', ascending=False)
results_df_sorted.to_csv('../reports/model_comparison_results.csv', index=False)
print("✅ Tabla de resultados guardada: reports/model_comparison_results.csv")

# Crear tabla detallada con métricas adicionales
detailed_results = []

for name in all_predictions.keys():
    y_pred = all_predictions[name]
    y_proba = all_probabilities[name]
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    detailed_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1_Score': f1_score(y_test, y_pred),
        'ROC_AUC': roc_auc_score(y_test, y_proba),
        'MCC': matthews_corrcoef(y_test, y_pred),
        'Specificity': tn / (tn + fp),
        'Sensitivity': tp / (tp + fn),
        'True_Positives': tp,
        'True_Negatives': tn,
        'False_Positives': fp,
        'False_Negatives': fn
    })

detailed_df = pd.DataFrame(detailed_results).sort_values('F1_Score', ascending=False)
detailed_df.to_csv('../reports/detailed_metrics.csv', index=False)
print("✅ Métricas detalladas guardadas: reports/detailed_metrics.csv")

# Mostrar tabla
print("\n📊 TOP 5 MODELOS - RANKING POR F1-SCORE:")
print("="*100)
print(detailed_df[['Model', 'F1_Score', 'ROC_AUC', 'Precision', 'Recall', 'MCC']].head().to_string(index=False))

## 6️⃣ Optimización de Hiperparámetros

Usamos GridSearchCV para encontrar los mejores hiperparámetros del modelo ganador.  
**Métrica de optimización**: F1-Score (balance entre Precision y Recall)

In [ ]:
# Seleccionar el mejor modelo basado en F1-Score
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]

print(f"🏆 Modelo seleccionado para optimización: {best_model_name}")
print(f"   F1-Score inicial: {results_df.iloc[0]['F1_Score']:.4f}")

# Grid de hiperparámetros según el mejor modelo
# Ajustar según el modelo que haya ganado en la evaluación anterior

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20],
    'min_samples_split': [10, 20, 30],
    'min_samples_leaf': [5, 10, 15]
}

# GridSearchCV con validación cruzada
print(f"\n🔍 Optimizando hiperparámetros con GridSearchCV...")
print(f"Grid: {sum(len(v) for v in param_grid.values())} combinaciones")

grid_search = GridSearchCV(
    best_model,                # Modelo con mejor F1-Score
    param_grid,
    cv=5,                      # 5-fold cross-validation
    scoring='f1',              # Optimizar F1-Score
    n_jobs=-1,                 # Usar todos los cores
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n✅ Optimización completada")
print(f"Mejores hiperparámetros: {grid_search.best_params_}")
print(f"Mejor F1-Score (CV): {grid_search.best_score_:.4f}")

## 7️⃣ Evaluación del Modelo Optimizado

Evaluamos el modelo con hiperparámetros óptimos en el conjunto de test.

In [ ]:
# Obtener el mejor modelo optimizado
best_optimized_model = grid_search.best_estimator_

# Predicciones en test
y_pred_final = best_optimized_model.predict(X_test)
y_proba_final = best_optimized_model.predict_proba(X_test)[:, 1]

# Calcular métricas finales
print("📊 MÉTRICAS FINALES DEL MODELO OPTIMIZADO")
print("="*80)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_final):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_final):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba_final):.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_final, 
                          target_names=['No Compra', 'Compra']))

## 8️⃣ Análisis de Feature Importance

Identificamos las variables más importantes para las predicciones del modelo.

In [ ]:
# Seleccionar modelo base según el mejor
if 'XGBoost' in best_model_name:
    base_model = XGBClassifier(random_state=42, eval_metric='logloss')
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [5, 7, 10],
        'learning_rate': [0.01, 0.1],
        'subsample': [0.8, 1.0]
    }
elif 'LightGBM' in best_model_name:
    base_model = LGBMClassifier(random_state=42, verbose=-1)
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [5, 7, 10],
        'learning_rate': [0.01, 0.1],
        'num_leaves': [31, 50]
    }
elif 'Random Forest' in best_model_name:
    base_model = RandomForestClassifier(random_state=42)
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, 15, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }
else:
    base_model = GradientBoostingClassifier(random_state=42)
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1]
    }

print(f"🔧 Optimizando hiperparámetros de {best_model_name}...")
print(f"   Parámetros a probar: {param_grid}")

# GridSearchCV
grid_search = GridSearchCV(
    base_model, param_grid, 
    cv=5, scoring='f1',
    n_jobs=-1, verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\n✅ Optimización completada")
print(f"   Mejores parámetros: {grid_search.best_params_}")
print(f"   Mejor F1-Score (CV): {grid_search.best_score_:.4f}")

In [ ]:
# Evaluar modelo optimizado
best_model_optimized = grid_search.best_estimator_

y_pred_opt = best_model_optimized.predict(X_test)
y_proba_opt = best_model_optimized.predict_proba(X_test)[:, 1]

print(f"\n📊 Resultados del Modelo Optimizado:")
print(f"{'='*80}")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_opt):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_opt):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_opt):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred_opt):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_proba_opt):.4f}")
print(f"  MCC:       {matthews_corrcoef(y_test, y_pred_opt):.4f}")

print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred_opt, target_names=['No Compra', 'Compra']))

# Matriz de confusión del modelo optimizado
cm_opt = confusion_matrix(y_test, y_pred_opt)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='RdYlGn', cbar=True, ax=ax,
            xticklabels=['No Compra', 'Compra'],
            yticklabels=['No Compra', 'Compra'])
ax.set_title(f'🏆 Matriz de Confusión - {best_model_name} Optimizado', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/06_best_model_confusion_matrix.png', dpi=300, bbox_inches='tight')
print("\n✅ Figura guardada: reports/figures/06_best_model_confusion_matrix.png")
plt.show()

In [ ]:
# Learning Curves del modelo optimizado
print("\n📈 Generando Learning Curves...")

train_sizes, train_scores, val_scores = learning_curve(
    best_model_optimized, X_train, y_train,
    cv=5, scoring='f1', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    random_state=42
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

fig, ax = plt.subplots(figsize=(12, 8))

# Plot train scores
ax.plot(train_sizes, train_mean, 'o-', color='#3498db', linewidth=3,
        label='Training F1-Score')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.2, color='#3498db')

# Plot validation scores
ax.plot(train_sizes, val_mean, 'o-', color='#e74c3c', linewidth=3,
        label='Validation F1-Score (CV)')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.2, color='#e74c3c')

ax.set_xlabel('Training Examples', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title(f'📊 Learning Curves - {best_model_name}', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/07_learning_curves.png', dpi=300, bbox_inches='tight')
print("✅ Figura guardada: reports/figures/07_learning_curves.png")
plt.show()

## 9️⃣ Importancia de Features

In [ ]:
# Obtener importancia de features
if hasattr(best_model_optimized, 'feature_importances_'):
    importances = best_model_optimized.feature_importances_
    
    # Crear DataFrame
    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    # Guardar CSV
    feature_importance_df.to_csv('../reports/feature_importance.csv', index=False)
    print("✅ Feature importance guardada: reports/feature_importance.csv")
    
    print("\n📊 Top 20 Features Más Importantes:")
    print("="*80)
    print(feature_importance_df.head(20).to_string(index=False))
    
    # Visualización - Top 20
    plt.figure(figsize=(12, 10))
    top_features = feature_importance_df.head(20)
    
    bars = plt.barh(range(len(top_features)), top_features['Importance'], 
                    color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
    plt.yticks(range(len(top_features)), top_features['Feature'])
    plt.xlabel('Importance Score', fontsize=12, fontweight='bold')
    plt.title(f'🔑 Top 20 Features Más Importantes - {best_model_name}', 
              fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    
    # Añadir valores
    for i, (bar, val) in enumerate(zip(bars, top_features['Importance'])):
        plt.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../reports/figures/08_feature_importance.png', dpi=300, bbox_inches='tight')
    print("\n✅ Figura guardada: reports/figures/08_feature_importance.png")
    plt.show()
    
    # Análisis acumulativo
    cumulative_importance = np.cumsum(feature_importance_df['Importance'].values)
    n_features_80 = np.argmax(cumulative_importance >= 0.80) + 1
    n_features_90 = np.argmax(cumulative_importance >= 0.90) + 1
    
    print(f"\n📈 Análisis de Importancia Acumulativa:")
    print(f"  - {n_features_80} features explican el 80% de la importancia")
    print(f"  - {n_features_90} features explican el 90% de la importancia")
    
    # Gráfica acumulativa
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(range(1, len(cumulative_importance)+1), cumulative_importance, 
            'b-', linewidth=2, label='Importancia Acumulativa')
    ax.axhline(y=0.80, color='r', linestyle='--', label='80% Threshold')
    ax.axhline(y=0.90, color='orange', linestyle='--', label='90% Threshold')
    ax.axvline(x=n_features_80, color='r', linestyle=':', alpha=0.5)
    ax.axvline(x=n_features_90, color='orange', linestyle=':', alpha=0.5)
    
    ax.set_xlabel('Number of Features', fontsize=12, fontweight='bold')
    ax.set_ylabel('Cumulative Importance', fontsize=12, fontweight='bold')
    ax.set_title('📊 Importancia Acumulativa de Features', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../reports/figures/09_cumulative_importance.png', dpi=300, bbox_inches='tight')
    print("✅ Figura guardada: reports/figures/09_cumulative_importance.png")
    plt.show()
    
else:
    print("⚠️ El modelo no proporciona importancia de features")

## 🔟 Guardar Modelo Final

In [ ]:
# Guardar modelo optimizado y scaler
model_filename = '../models/best_model.pkl'
scaler_filename = '../models/scaler.pkl'
model_info_filename = '../models/model_info.pkl'

joblib.dump(best_model_optimized, model_filename)
print(f"✅ Modelo guardado: {model_filename}")

# Cargar y guardar scaler del preprocesamiento
scaler = joblib.load('../data/02_processed/scaler.pkl')
joblib.dump(scaler, scaler_filename)
print(f"✅ Scaler guardado: {scaler_filename}")

# Guardar información completa del modelo
model_info = {
    'model_name': best_model_name,
    'model_type': type(best_model_optimized).__name__,
    'best_params': grid_search.best_params_,
    'cv_best_score': grid_search.best_score_,
    'test_accuracy': accuracy_score(y_test, y_pred_opt),
    'test_precision': precision_score(y_test, y_pred_opt),
    'test_recall': recall_score(y_test, y_pred_opt),
    'test_f1': f1_score(y_test, y_pred_opt),
    'test_roc_auc': roc_auc_score(y_test, y_proba_opt),
    'test_mcc': matthews_corrcoef(y_test, y_pred_opt),
    'feature_names': feature_names,
    'n_features': len(feature_names),
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'train_size': len(X_train),
    'test_size': len(X_test)
}

joblib.dump(model_info, model_info_filename)
print(f"✅ Información del modelo guardada: {model_info_filename}")

# Crear reporte de texto
report_filename = '../reports/model_training_report.txt'
with open(report_filename, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("📊 REPORTE DE ENTRENAMIENTO DE MODELO\n")
    f.write("Online Shoppers Purchasing Intention Dataset\n")
    f.write("="*80 + "\n\n")
    
    f.write(f"📅 Fecha de entrenamiento: {model_info['training_date']}\n")
    f.write(f"👤 Autor: Miguel Antonio Benítez González\n")
    f.write(f"📧 Email: mbenitezg01@gmail.com\n\n")
    
    f.write("="*80 + "\n")
    f.write("🤖 MODELOS EVALUADOS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Total de modelos entrenados: {len(models)}\n\n")
    for name in models.keys():
        f.write(f"  • {name}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("🏆 MEJOR MODELO\n")
    f.write("="*80 + "\n\n")
    f.write(f"Modelo seleccionado: {best_model_name}\n")
    f.write(f"Tipo: {model_info['model_type']}\n\n")
    
    f.write("Hiperparámetros optimizados:\n")
    for param, value in grid_search.best_params_.items():
        f.write(f"  • {param}: {value}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("📊 MÉTRICAS DE RENDIMIENTO (Conjunto de Prueba)\n")
    f.write("="*80 + "\n\n")
    f.write(f"Accuracy:             {model_info['test_accuracy']:.4f}\n")
    f.write(f"Precision:            {model_info['test_precision']:.4f}\n")
    f.write(f"Recall (Sensitivity): {model_info['test_recall']:.4f}\n")
    f.write(f"F1-Score:             {model_info['test_f1']:.4f}\n")
    f.write(f"ROC-AUC:              {model_info['test_roc_auc']:.4f}\n")
    f.write(f"MCC:                  {model_info['test_mcc']:.4f}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("📈 INFORMACIÓN DE DATOS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Tamaño conjunto entrenamiento: {model_info['train_size']:,} muestras\n")
    f.write(f"Tamaño conjunto prueba:        {model_info['test_size']:,} muestras\n")
    f.write(f"Número de features:            {model_info['n_features']}\n")
    
    if hasattr(best_model_optimized, 'feature_importances_'):
        f.write("\n" + "="*80 + "\n")
        f.write("🔑 TOP 10 FEATURES MÁS IMPORTANTES\n")
        f.write("="*80 + "\n\n")
        for i, row in feature_importance_df.head(10).iterrows():
            f.write(f"{i+1:2d}. {row['Feature']:30s} {row['Importance']:.6f}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("📁 ARCHIVOS GENERADOS\n")
    f.write("="*80 + "\n\n")
    f.write("Modelos:\n")
    f.write("  • models/best_model.pkl\n")
    f.write("  • models/scaler.pkl\n")
    f.write("  • models/model_info.pkl\n\n")
    
    f.write("Reportes:\n")
    f.write("  • reports/model_comparison_results.csv\n")
    f.write("  • reports/detailed_metrics.csv\n")
    f.write("  • reports/feature_importance.csv\n")
    f.write("  • reports/model_training_report.txt\n\n")
    
    f.write("Figuras:\n")
    f.write("  • reports/figures/01_model_comparison.png\n")
    f.write("  • reports/figures/02_overfitting_analysis.png\n")
    f.write("  • reports/figures/03_confusion_matrices.png\n")
    f.write("  • reports/figures/04_roc_curves.png\n")
    f.write("  • reports/figures/05_precision_recall_curves.png\n")
    f.write("  • reports/figures/06_best_model_confusion_matrix.png\n")
    f.write("  • reports/figures/07_learning_curves.png\n")
    f.write("  • reports/figures/08_feature_importance.png\n")
    f.write("  • reports/figures/09_cumulative_importance.png\n\n")
    
    f.write("="*80 + "\n")
    f.write("✅ ENTRENAMIENTO COMPLETADO CON ÉXITO\n")
    f.write("="*80 + "\n")

print(f"✅ Reporte guardado: {report_filename}")

## 1️⃣1️⃣ Resumen del Modelado

In [ ]:
print("="*100)
print("📝 RESUMEN FINAL DEL MODELADO")
print("="*100)

print(f"\n🤖 MODELOS ENTRENADOS Y EVALUADOS: {len(models)}")
for name in models.keys():
    f1 = results_df[results_df['Model']==name]['F1_Score'].values[0]
    print(f"  • {name:25s} - F1: {f1:.4f}")

print(f"\n🏆 MEJOR MODELO SELECCIONADO: {best_model_name}")
print(f"   Tipo: {type(best_model_optimized).__name__}")
print(f"   Parámetros optimizados: {grid_search.best_params_}")

print(f"\n📊 MÉTRICAS FINALES EN CONJUNTO DE PRUEBA:")
print(f"  ┌{'─'*50}┐")
print(f"  │ {'Métrica':<25s} {'Valor':>20s} │")
print(f"  ├{'─'*50}┤")
print(f"  │ {'Accuracy':<25s} {accuracy_score(y_test, y_pred_opt):>20.4f} │")
print(f"  │ {'Precision':<25s} {precision_score(y_test, y_pred_opt):>20.4f} │")
print(f"  │ {'Recall':<25s} {recall_score(y_test, y_pred_opt):>20.4f} │")
print(f"  │ {'F1-Score':<25s} {f1_score(y_test, y_pred_opt):>20.4f} │")
print(f"  │ {'ROC-AUC':<25s} {roc_auc_score(y_test, y_proba_opt):>20.4f} │")
print(f"  │ {'MCC':<25s} {matthews_corrcoef(y_test, y_pred_opt):>20.4f} │")
print(f"  └{'─'*50}┘")

if hasattr(best_model_optimized, 'feature_importances_'):
    print(f"\n🔝 TOP 5 FEATURES MÁS IMPORTANTES:")
    for idx, row in feature_importance_df.head(5).iterrows():
        print(f"  {idx+1}. {row['Feature']:30s} - {row['Importance']:.6f}")

print(f"\n💾 ARCHIVOS GENERADOS:")
print(f"  📦 Modelos:")
print(f"     ✅ models/best_model.pkl")
print(f"     ✅ models/scaler.pkl")
print(f"     ✅ models/model_info.pkl")

print(f"\n  📊 Reportes y Datos:")
print(f"     ✅ reports/model_comparison_results.csv")
print(f"     ✅ reports/detailed_metrics.csv")
print(f"     ✅ reports/feature_importance.csv")
print(f"     ✅ reports/model_training_report.txt")

print(f"\n  📈 Figuras:")
print(f"     ✅ 01_model_comparison.png")
print(f"     ✅ 02_overfitting_analysis.png")
print(f"     ✅ 03_confusion_matrices.png")
print(f"     ✅ 04_roc_curves.png")
print(f"     ✅ 05_precision_recall_curves.png")
print(f"     ✅ 06_best_model_confusion_matrix.png")
print(f"     ✅ 07_learning_curves.png")
print(f"     ✅ 08_feature_importance.png")
print(f"     ✅ 09_cumulative_importance.png")

print(f"\n🚀 SIGUIENTE PASO:")
print(f"  🌐 API REST lista en: api/main.py")
print(f"  🖥️ Dashboard listo en: web/app.py")
print(f"  🐳 Docker configurado en: docker/")

print("\n" + "="*100)
print("✅ ENTRENAMIENTO Y ANÁLISIS COMPLETADOS CON ÉXITO")
print("="*100)

---

## ✅ Resumen del Modelado

### 🤖 Modelos Evaluados

| Modelo | Tipo | Características |
|--------|------|-----------------|
| **Logistic Regression** | Lineal | Baseline, interpretable |
| **Decision Tree** | Árbol | No lineal, rápido |
| **Random Forest** | Ensemble | Bagging, robusto |
| **Gradient Boosting** | Ensemble | Boosting, preciso |
| **XGBoost** | Ensemble | Optimizado, popular |
| **LightGBM** | Ensemble | Rápido, eficiente |

### 🏆 Mejor Modelo

El modelo con mejor rendimiento fue seleccionado basándose en:
- **Métrica principal**: F1-Score (balance Precision-Recall)
- **Validación**: 5-fold cross-validation
- **Optimización**: GridSearchCV para hiperparámetros

### 📊 Métricas de Evaluación

Las métricas en el conjunto de prueba demuestran:
- Balance entre Precision y Recall apropiado para el problema
- ROC-AUC alto indicando buena capacidad de discriminación
- Buen rendimiento en clase minoritaria (Compra)

### 🔑 Features Más Importantes

Top variables identificadas por el modelo:
1. **PageValues** - Valor de las páginas visitadas
2. **ExitRates** - Tasa de salida de sesión
3. **ProductRelated_Duration** - Tiempo en páginas de productos
4. **EngagementScore** - Score creado por feature engineering
5. **ProductRatio** - Proporción de páginas de productos

**Insight**: Las variables de engagement y métricas de Google Analytics son las más predictivas.

### 💾 Artefactos Guardados

```
models/
├── best_model.pkl           # Modelo optimizado entrenado
├── scaler.pkl               # StandardScaler para preprocesamiento
└── model_info.pkl           # Metadata: métricas, features, hiperparámetros
```

### 🎯 Rendimiento del Modelo

El modelo final está listo para:
- ✅ Predicciones en tiempo real via API
- ✅ Integración en sistema de producción
- ✅ Scoring de visitantes nuevos
- ✅ Reentrenamiento con datos actualizados

### 🚀 Próximos Pasos

1. **Desarrollo de API REST** (`api/main.py`)
   - Endpoints para predicción individual y batch
   - Preprocesamiento idéntico al notebook
   - Documentación automática con Swagger

2. **Interfaz Web** (`web/app.py`)
   - Dashboard interactivo con Streamlit
   - Formulario de predicción
   - Visualización de probabilidades

3. **Deployment con Docker**
   - Containerización completa
   - Orquestación con docker-compose
   - Health checks y logging

---

## 💡 Lecciones Aprendidas

### ✅ Buenas Prácticas Aplicadas

1. **Balanceo solo en Train**: Test mantiene distribución real
2. **Feature Engineering**: Variables derivadas mejoran el modelo
3. **Validación Cruzada**: Evita overfitting
4. **Múltiples Métricas**: No solo Accuracy en datasets desbalanceados
5. **Reproducibilidad**: random_state=42 en todos los pasos

### ⚠️ Consideraciones para Producción

- **Drift Detection**: Monitorear cambio en distribución de datos
- **Re-entrenamiento**: Actualizar modelo periódicamente
- **A/B Testing**: Validar mejoras en producción
- **Explicabilidad**: SHAP values para interpretar predicciones

---

**🎯 Proyecto Completo**: EDA → Preprocesamiento → Modelado → API → Web → Docker